# Chapter 21 — Federated Threat Intelligence — 사내 데이터 보호 공동 학습

> **Chapter 21 · Part III**

교육용 위협 문장을 임베딩해 가우시안 노이즈가 검색 유용성(Recall@k)과 최근접 원본 추정에 주는 영향을 관찰합니다. 같은 원본에 넣은 독립 노이즈를 평균하는 예제로 유용성 변화와 반복 공개의 한계를 살펴봅니다.

## 실습 목표
1. 민감도 2를 기준으로 계산한 가우시안 노이즈를 문장 임베딩에 더합니다.
2. ε 별로 검색 Recall@k(유용성) 하락과 재식별 난이도(원본 근접도)를 **여러 번 평균**으로 함께 측정해 트레이드오프 곡선을 그립니다.
3. 측정된 원본 추정 성공률과 수학적 DP 보장을 구분합니다.
4. 독립 노이즈를 평균할 때의 유용성 변화를 관찰하고 누적 예산 문제를 설명합니다.

> **주의:** `sentence-transformers` 로 임베딩을 만들고(Colab 에 사전 설치), 나머지는 numpy 로 동작합니다. 위협 문장은 교육용 예시이며 실제 사고 데이터가 아닙니다.

In [1]:
# Colab: sentence-transformers 는 대개 사전 설치돼 있으나, 없을 수 있어 명시 설치
# [검증 실행] 생략됨: !pip install -q sentence-transformers numpy
import numpy as np, math

In [2]:
from sentence_transformers import SentenceTransformer

# 교육용 위협 인텔 문장 corpus (실제 사고 데이터 아님)
CORPUS = [
    "다중 실패 로그인 후 관리자 계정 탈취", "피싱 메일 첨부파일 매크로 실행",
    "C2 서버로 주기적 비콘 통신", "내부망 측면 이동 시도 탐지",
    "랜섬웨어 파일 대량 암호화", "권한 상승 익스플로잇 사용",
    "DNS 터널링으로 데이터 유출", "웹셸 업로드 후 원격 명령 실행",
    "SSH 포트 무차별 대입 공격", "크리덴셜 스터핑 로그인 폭주",
    "SQL 인젝션 페이로드 탐지", "제로데이 취약점 스캐닝",
    "봇넷 감염 호스트 다수 발견", "임원 표적 스피어피싱 메일",
    "공급망 업데이트 파일 변조", "루트킷 커널 모듈 로드",
    "데이터 암호화 후 몸값 요구", "VPN 계정 비정상 국가 접속",
    "클라우드 액세스 키 유출 의심", "프로세스 할로잉 메모리 인젝션",
    "APT 그룹 지속 접근 유지", "방화벽 우회 포트 스캔",
    "악성 브라우저 확장 프로그램 설치", "세션 토큰 탈취 하이재킹",
    "비인가 USB 저장장치 연결", "로그 삭제로 흔적 은폐 시도",
    "비정상 대용량 아웃바운드 트래픽", "가짜 업데이트 서버 리다이렉트",
    "관리자 권한 스케줄 작업 등록", "메일 규칙 조작 자동 전달 설정",
]

model = SentenceTransformer('all-MiniLM-L6-v2')
# normalize_embeddings=True → 단위 L2 노름. 벡터 교체 인접성의 L2 민감도 상한은 2.0(본문 §21.3.2)
E = model.encode(CORPUS, normalize_embeddings=True).astype(np.float64)
N_DOCS, DIM = E.shape
print(f'corpus {N_DOCS} 문장, 임베딩 차원 {DIM}, 노름 예시 {np.linalg.norm(E[0]):.3f}')

<python-packages>/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


corpus 30 문장, 임베딩 차원 384, 노름 예시 1.000


이 노트북은 고정 난수 시드로 노이즈와 검색의 관계를 재현하는 시뮬레이션입니다. 실제 개인정보에 사용할 DP 구현이 아닙니다. ε ≥ 1은 노이즈 스케일 비교만 뜻합니다. 여러 조직이 동일한 E를 공개하는 예제는 독립 개인정보 집계가 아니며, 반복 공개의 누적 프라이버시 손실을 따로 계산해야 합니다.

---
## Step 1. DP 가우시안 메커니즘

DP 가우시안 메커니즘의 노이즈 크기를 고정 시드 예제에 적용해 봅니다. 고전적 가우시안
메커니즘의 노이즈 표준편차는

    sigma > sensitivity * sqrt(2 · ln(1.25/δ)) / ε

조건을 만족하도록 정합니다(코드는 하한의 1.01배). ε이 작을수록 노이즈가 커집니다. 두 가지를 분명히 해 둡니다.

- 이 닫힌형 하한은 Dwork·Roth 의 고전적 결과로 **ε ∈ (0, 1) 에서만 엄밀한 DP 보장**입니다.
  아래에서 비교를 위해 ε ≥ 1 도 쓰지만, 그 구간에서는 "ε 에 반비례하는 노이즈 스케일"로만
  해석하고 정식 보장은 analytic Gaussian mechanism·Opacus 같은 회계 도구로 계산해야 합니다.
- `sigma = sensitivity/epsilon` 은 **Laplace 스케일이지 가우시안 표준편차가 아닙니다**(본문 §21.3.2).
  단위 노름 벡터 한 개를 교체하는 인접성에서는 민감도 상한을 2.0으로 둡니다.

In [3]:
rng = np.random.default_rng(42)

def dp_sigma(epsilon, delta=1e-5, sensitivity=2.0):
    """고전적 가우시안 메커니즘의 노이즈 표준편차. ε∈(0,1) 에서 엄밀."""
    return 1.01 * sensitivity * math.sqrt(2 * math.log(1.25 / delta)) / epsilon

def add_dp_noise_gaussian(mat, epsilon, delta=1e-5, sensitivity=2.0):
    """행별 임베딩에 (ε, δ)-DP 가우시안 노이즈를 더한다."""
    sigma = dp_sigma(epsilon, delta, sensitivity)
    return mat + rng.normal(0, sigma, mat.shape)

# ε 이 작을수록 sigma(노이즈) 가 커진다
for eps in (0.1, 1.0, 5.0):
    print(f'ε={eps:4}: sigma={dp_sigma(eps):6.2f}  (임베딩 성분 스케일 약 {1/math.sqrt(DIM):.3f})')

ε= 0.1: sigma= 97.87  (임베딩 성분 스케일 약 0.051)
ε= 1.0: sigma=  9.79  (임베딩 성분 스케일 약 0.051)
ε= 5.0: sigma=  1.96  (임베딩 성분 스케일 약 0.051)


---
## Step 2. 유용성(Recall@k) 과 재식별 난이도 측정 — ε 트레이드오프

노이즈를 넣은 임베딩을 hub 에 공유한다고 할 때, 두 가지를 동시에 잽니다.

- **유용성(Recall@k)**: 내 원본 질의로 노이즈 인덱스를 검색해 맞는 문서가 상위 k 안에 드는 비율.
- **재식별(top-1)**: 공개된 노이즈 임베딩이 어느 원본에 가장 가까운지 골랐을 때 실제 원본을 맞히는 비율(이 최근접 추정 방식의 성공률이며 일반적인 개인정보 보호 보장이 아닙니다).

한 번의 무작위 추출이 아니라 여러 번(trials) 돌려 평균합니다.

In [4]:
def recall_at_k(noisy_index, k=5):
    """원본 임베딩 E 로 노이즈 인덱스를 검색해 자기 문서가 상위 k 에 드는 비율(유용성)."""
    sims = E @ noisy_index.T                      # 노이즈 벡터와의 내적 점수(코사인 유사도가 아님)
    return np.mean([i in np.argsort(-sims[i])[:k] for i in range(N_DOCS)])

def reid_top1(noisy_shared):
    """공개된 노이즈 임베딩이 실제 원본과 최근접인 비율(재식별 성공률, 낮을수록 강한 보호)."""
    sims = noisy_shared @ E.T
    return np.mean(np.argmax(sims, axis=1) == np.arange(N_DOCS))

def measure(epsilon, trials=30, k=5):
    recs, reids = [], []
    for _ in range(trials):
        noisy = add_dp_noise_gaussian(E, epsilon)
        recs.append(recall_at_k(noisy, k))
        reids.append(reid_top1(noisy))
    return float(np.mean(recs)), float(np.mean(reids))

EPS_GRID = [0.1, 1.0, 5.0, 10.0, 20.0, 50.0]
results = {eps: measure(eps) for eps in EPS_GRID}

print(f"{'ε':>6}  {'sigma':>7}  {'Recall@5(유용성)':>16}  {'재식별top1(공격)':>16}")
print('-' * 56)
for eps in EPS_GRID:
    rec, reid = results[eps]
    print(f"{eps:6}  {dp_sigma(eps):7.2f}  {rec:16.2f}  {reid:16.2f}")
print('\nε 이 커질수록(노이즈↓): 검색은 살아나지만(유용성↑) 재식별도 쉬워집니다(보호↓) — 정면 트레이드오프.')

     ε    sigma     Recall@5(유용성)       재식별top1(공격)
--------------------------------------------------------
   0.1    97.87              0.18              0.05
   1.0     9.79              0.16              0.03
   5.0     1.96              0.24              0.07
  10.0     0.98              0.30              0.11
  20.0     0.49              0.46              0.26
  50.0     0.20              0.79              0.75

ε 이 커질수록(노이즈↓): 검색은 살아나지만(유용성↑) 재식별도 쉬워집니다(보호↓) — 정면 트레이드오프.


---
## Step 3. 최근접 원본 추정 성공률의 해석

그림의 보호 관련 라벨은 이 데이터와 최근접 추정 방식에서의 관찰값입니다.
다른 공격자의 성공률이나 수학적 DP 보장을 대신하지 않습니다.
DP 보장은 인접성·민감도·난수·반복 공개의 누적 예산으로 검토해야 합니다.


In [5]:
def protection_label(reid_rate):
    # 무작위 추측 기준선 = 1/N_DOCS. 재식별 성공률로 보호 강도를 라벨링
    if reid_rate <= 3 / N_DOCS:
        return '강한 보호(재식별 거의 무작위 수준)'
    if reid_rate <= 0.35:
        return '중간 보호'
    return '약한 보호(재식별 쉬움)'

print(f'무작위 추측 재식별률 기준선: {1/N_DOCS:.3f}\n')
print(f"{'ε':>6}  {'재식별top1':>10}  {'Recall@5':>9}  보호 강도(측정 기반)")
print('-' * 60)
for eps in EPS_GRID:
    rec, reid = results[eps]
    print(f"{eps:6}  {reid:10.2f}  {rec:9.2f}  {protection_label(reid)}")

무작위 추측 재식별률 기준선: 0.033

     ε     재식별top1   Recall@5  보호 강도(측정 기반)
------------------------------------------------------------
   0.1        0.05       0.18  강한 보호(재식별 거의 무작위 수준)
   1.0        0.03       0.16  강한 보호(재식별 거의 무작위 수준)
   5.0        0.07       0.24  강한 보호(재식별 거의 무작위 수준)
  10.0        0.11       0.30  중간 보호
  20.0        0.26       0.46  중간 보호
  50.0        0.75       0.79  약한 보호(재식별 쉬움)


---
## Step 4. 독립 노이즈 평균과 누적 공개

같은 원본 E에 독립 노이즈를 K번 더한 후 평균합니다. 노이즈 분산은 1/K로 줄고
검색 유용성이 회복될 수 있습니다. 하지만 같은 개인정보의 반복 공개라면
프라이버시 예산이 누적되므로, 원래의 ε로 보호된다고 해석할 수 없습니다.
이 예제는 secure aggregation이나 서로 독립된 조직의 실제 데이터를 구현하지 않습니다.


In [6]:
def federated_aggregate(epsilon, n_orgs, trials=30, k=5):
    """n_orgs 개 조직이 각자 per-조직 ε 로 노이즈를 넣어 기여하고 hub 가 평균낸다."""
    recs, reids = [], []
    for _ in range(trials):
        contribs = [add_dp_noise_gaussian(E, epsilon) for _ in range(n_orgs)]
        agg = np.mean(contribs, axis=0)
        recs.append(recall_at_k(agg, k))
        reids.append(reid_top1(agg))
    return float(np.mean(recs)), float(np.mean(reids))

EPS_PER_ORG = 5.0   # 설명용 노이즈 계수: 이 값은 고전식의 DP 보장 범위 밖
print(f'per-조직 ε={EPS_PER_ORG} 고정, 조직 수(K)만 늘려 hub 집계\n')
print(f"{'K조직':>6}  {'실효sigma':>9}  {'Recall@5':>9}  {'재식별top1':>10}")
print('-' * 42)
for K in (1, 3, 5, 10):
    rec, reid = federated_aggregate(EPS_PER_ORG, K)
    print(f"{K:6}  {dp_sigma(EPS_PER_ORG)/math.sqrt(K):9.2f}  {rec:9.2f}  {reid:10.2f}")
print('\n조직이 늘수록 같은 per-조직 ε 에서도 집계 유용성이 회복됩니다(노이즈가 평균으로 상쇄).')

per-조직 ε=5.0 고정, 조직 수(K)만 늘려 hub 집계

   K조직    실효sigma   Recall@5     재식별top1
------------------------------------------
     1       1.96       0.22        0.07
     3       1.13       0.29        0.10
     5       0.88       0.30        0.13
    10       0.62       0.37        0.19

조직이 늘수록 같은 per-조직 ε 에서도 집계 유용성이 회복됩니다(노이즈가 평균으로 상쇄).


---
## 정리
- 고정 시드 교육 예제로 노이즈 크기와 검색 유용성의 관계를 관찰했습니다.
- 최근접 원본 추정 성공률은 제한된 공격의 관찰값이며, PII 노출 방지나 DP의 증명이 아닙니다.
- 같은 원본의 독립 노이즈를 평균하면 유용성이 회복될 수 있지만 반복 공개의 예산이 누적됩니다.
- 실제 적용에는 비공개 난수, 인접성·민감도 정의와 검증된 메커니즘·회계가 필요합니다.
  ε ≥ 1 구간은 이 고전적 공식의 DP 보장으로 해석하지 않습니다.

> **다음 챕터:** LLM Red Team 자동화 도구.
